# Clumps (Rigid Bodies)

A **clump** is a rigid body assembled from multiple spheres that share the same
:py:attr:`~jaxdem.state.State.clump_id`. Every sphere in a clump stores the
same body-level fields (COM position, orientation, velocity, angular velocity,
total mass, total inertia, aggregated force/torque) while keeping its own
body-frame offset :py:attr:`~jaxdem.state.State.pos_p`, radius, and
material/species IDs.


## The Clump Data Model
.. list-table::
   :header-rows: 1

   * - Field
     - Shared by clump
     - Per-sphere
   * - :py:attr:`~jaxdem.state.State.pos_c`, :py:attr:`~jaxdem.state.State.q`, :py:attr:`~jaxdem.state.State.vel`, :py:attr:`~jaxdem.state.State.ang_vel`, :py:attr:`~jaxdem.state.State.fixed`
     - ✓
     -
   * - :py:attr:`~jaxdem.state.State.force`, :py:attr:`~jaxdem.state.State.torque`, :py:attr:`~jaxdem.state.State.mass`, :py:attr:`~jaxdem.state.State.inertia`
     - ✓
     -
   * - :py:attr:`~jaxdem.state.State.pos_p`, :py:attr:`~jaxdem.state.State.rad`, :py:attr:`~jaxdem.state.State.volume`
     -
     - ✓
   * - :py:attr:`~jaxdem.state.State.clump_id`, :py:attr:`~jaxdem.state.State.bond_id`, :py:attr:`~jaxdem.state.State.mat_id`, :py:attr:`~jaxdem.state.State.species_id`
     -
     - ✓

Lab-frame sphere positions are computed on access from COM and body offsets:

\begin{align}\texttt{pos} = \texttt{pos\_c} + R(q)\,\texttt{pos\_p}\end{align}

Note that both :py:meth:`~jaxdem.state.State.create` and
:py:meth:`~jaxdem.state.State.add_clump` take the center of mass via the
``pos=`` parameter (which populates ``pos_c``) and offsets via ``pos_p=``.



In [ ]:
import jax.numpy as jnp

import jaxdem as jdem

## Creating Clumps Manually and with :py:meth:`~jaxdem.state.State.add_clump`
Build a dumbbell clump (clump 0) beside a free sphere (clump 1):



In [ ]:
state = jdem.State.create(
    pos=jnp.array([[2.0, 0.0], [2.0, 0.0], [6.0, 0.0]]),  # COM (pos_c)
    pos_p=jnp.array([[-0.5, 0.0], [0.5, 0.0], [0.0, 0.0]]),
    rad=jnp.array([0.6, 0.6, 1.0]),
    clump_id=jnp.array([0, 0, 1]),
)
print("clump_id:", state.clump_id)
print("lab pos :\n", state.pos)

# Keep this construction snapshot for the initial ParaView illustration.
scene_preview_state = state
scene_preview_system = jdem.System.create(state=state, collider_type="naive")

.. only:: jaxdem_visuals

   .. simulation-image:: /_static/visuals/clump_guide/initial.png
      :alt: Two overlapping member disks forming one dumbbell clump beside a separate sphere.
      :width: 100%

   The two overlapping disks share one clump ID and move as a rigid body beside
   a single sphere.



:py:meth:`~jaxdem.state.State.add_clump` appends a new clump to an existing
state, broadcasts shared fields (``vel``, ``mass``, ``volume``, …) to all
members, and assigns a fresh ``clump_id``:



In [ ]:
state_base = jdem.State.create(pos=jnp.array([[0.0, 0.0]]), rad=jnp.array([1.0]))
state_with_clump = jdem.State.add_clump(
    state_base,
    pos=jnp.array([[3.0, 0.0], [3.0, 0.0]]),
    pos_p=jnp.array([[-0.4, 0.0], [0.4, 0.0]]),
    rad=jnp.array([0.5, 0.5]),
    vel=jnp.array([1.0, 0.0]),
    mass=jnp.array(2.0),
)
print("After add_clump: N =", state_with_clump.N, "| clump_ids =", state_with_clump.clump_id)

## Computing Union Mass, COM, and Principal Inertia
Member spheres inside a clump often overlap, so summing individual sphere
masses would double-count shared volume.
:py:func:`~jaxdem.utils.clumps.compute_clump_properties` computes the exact
material union $\Omega$ (analytically in 2D and for simple 3D overlaps,
and via adaptive quadrature controlled by ``n_samples`` for general 3D
overlaps) and diagonalizes the inertia tensor into principal axes while
keeping every sphere at the same lab-frame position:

\begin{align}M = \int_\Omega \rho\,dV, \quad
   \mathbf{c} = \frac{1}{M}\int_\Omega \rho\,\mathbf{x}\,dV, \quad
   \mathbf{I} = \int_\Omega \rho\left(\lVert\mathbf{x}-\mathbf{c}\rVert^2\mathbf{1}
   - (\mathbf{x}-\mathbf{c})(\mathbf{x}-\mathbf{c})^T\right)dV.\end{align}

Call it once during host-side setup (outside ``jax.jit``). For complex 3D
clumps, check convergence by increasing ``n_samples`` (e.g. ``50_000`` →
``200_000``). For ready-made multi-sphere shapes with ``core_type`` options
(``"hollow"``, ``"solid"``, ``"phantom"``), see
:doc:`../auto_examples/clump_construction_pipeline`.



In [ ]:
mat_table = jdem.MaterialTable.from_materials(
    [jdem.Material.create("elastic", density=2.0, young=1e4, poisson=0.3)]
)

# 2D overlapping dumbbell:
clump_2d = jdem.State.create(
    pos=jnp.array([[0.0, 0.0], [0.8, 0.0]]),
    rad=jnp.array([0.5, 0.5]),
    clump_id=jnp.array([0, 0]),
    mat_table=mat_table,
)
clump_2d = jdem.utils.compute_clump_properties(clump_2d, mat_table)
print("2D clump: COM =", clump_2d.pos_c[0], "| mass =", float(clump_2d.mass[0]), "| Izz =", clump_2d.inertia[0])

# 3D overlapping trimer (inertia has shape (N, 3) principal moments):
clump_3d = jdem.State.create(
    pos=jnp.array([[0.0, 0.0, 0.0], [0.7, 0.0, 0.0], [0.35, 0.6, 0.0]]),
    rad=jnp.full(3, 0.5),
    clump_id=jnp.zeros(3, dtype=int),
    mat_table=mat_table,
)
clump_3d = jdem.utils.compute_clump_properties(clump_3d, mat_table, n_samples=50_000)
print("3D clump: mass =", float(clump_3d.mass[0]), "| principal inertia =", clump_3d.inertia[0])

Because ``mass`` and ``inertia`` are broadcast to every member slot, select
one representative per clump (e.g. via ``jnp.unique`` or dividing by member
counts) when computing body-level totals such as total mass or kinetic energy:



In [ ]:
counts = jnp.bincount(state.clump_id, length=state.N)[state.clump_id]
total_mass = jnp.sum(state.mass / counts)
print("Total mass without double-counting members:", float(total_mass))

## Collisions, Force Aggregation, and Integration
* **Same-clump exclusion:** All colliders automatically skip pairs with the
  same :py:attr:`~jaxdem.state.State.clump_id` (unlike
  :py:attr:`~jaxdem.state.State.bond_id`, which only masks bonded vertices in
  deformable meshes — see :doc:`../auto_examples/deformable_particle_guide`).
* **Force & torque aggregation:** :py:class:`~jaxdem.forces.force_manager.ForceManager`
  adds lever-arm torques $\mathbf{r}_p \times \mathbf{F}$, sums forces
  and torques per ``clump_id`` via ``segment_sum``, and broadcasts the totals
  back to every member.
* **Integration & walls:** Integrators update shared COM fields directly, and
  :py:class:`~jaxdem.domains.reflect.ReflectDomain` aggregates wall-impact
  impulses across the whole clump.



## Running a Clump Impact Simulation
Drop a tilted dumbbell onto a fixed spherical floor under gravity. Asymmetric
impact transfers both linear rebound and angular velocity to the rigid body:



In [ ]:
state_sim = jdem.State.create(
    pos=jnp.array([[0.0, 0.0]]),
    rad=jnp.array([1.0]),
    fixed=jnp.array([True]),
    mat_table=mat_table,
)
state_sim = jdem.State.add_clump(
    state_sim,
    pos=jnp.array([[0.0, 4.0], [0.0, 4.0]]),
    pos_p=0.4 * jnp.array([[-jnp.cos(0.3), -jnp.sin(0.3)], [jnp.cos(0.3), jnp.sin(0.3)]]),
    rad=jnp.array([0.4, 0.4]),
)
state_sim = jdem.utils.compute_clump_properties(state_sim, mat_table)

system_sim = jdem.System.create(
    state_sim.shape,
    dt=1e-3,
    force_model_type="spring",
    collider_type="naive",
    mat_table=mat_table,
    force_manager_kw={"gravity": jnp.array([0.0, -9.81])},
)
state_sim, system_sim = jdem.System.initialize(state_sim, system_sim)
state_sim, system_sim, (scene_states, scene_systems) = jdem.System.trajectory_rollout(
    state_sim, system_sim, strides=jnp.asarray([0] + [40] * 40)
)
system_sim.check_overflow()
print("Final dumbbell COM:", state_sim.pos_c[1])
print("Peak upward rebound speed:", float(scene_states.vel[:, 1, 1].max()))
print("Peak angular speed:", float(jnp.abs(scene_states.ang_vel[:, 1]).max()))

## Common Pitfalls
1. **Skipping** :py:func:`~jaxdem.utils.clumps.compute_clump_properties`:
   manual clumps keep single-sphere mass and inertia until you compute union
   properties.
2. **Accidental duplicate ``clump_id``:** two separate bodies with the same
   ID fuse into one non-colliding rigid body.
3. **Non-contiguous ``clump_id`` after manual edits:** aggregation indexes
   ``N`` segments, so IDs must stay in ``[0, N)``.
4. **Mismatched shared fields:** if editing ``vel``, ``pos_c``, or ``q``
   manually, update every member slot of the clump identically (or use
   :py:meth:`~jaxdem.state.State.add_clump`).

